In [35]:
# from comet_ml import Experiment
# from comet_ml.integration.pytorch import log_model

import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn, optim
import torch.nn.functional as F
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import pickle as pkl

import matplotlib.pyplot as plt
import numpy as np
import io, os
from tqdm import tqdm

from PIL import Image
from torchvision import models 
from torchvision.models import resnet18

class ImageSequenceDataset(Dataset):
    def __init__(self, curve_dict, target_df,img_directory = 'data/curve_imgs/', split = 'train', sequence_len=40, 
                 mean=0, std=1):
        self.curve_dict = curve_dict
        self.target_df = target_df
        self.img_directory = img_directory
        self.split = split
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sequence_len = sequence_len

        self.mean = mean
        self.std = std

        # Image transformations: Resize and Normalize
        self.img_transforms = transforms.Compose([
            transforms.Lambda(lambda image: image.convert('RGB')),
            transforms.Resize((128, 128)),  # Resizing to a consistent size
            transforms.ToTensor(),  # Convert PIL image to tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalizing to [0,1]
            ])

        #Implementation of train test split
        if self.split == 'train':
            self.target_df = self.target_df[self.target_df['split']=='train']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'val':
            self.target_df = self.target_df[self.target_df['split']=='val']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        elif self.split == 'test':
            self.target_df = self.target_df[self.target_df['split']=='test']
            self.curve_dict = {k: self.curve_dict[k] for k in self.curve_dict.keys() if k in self.target_df['curve_idx'].values}
        else:
            raise NotImplementedError
        
    def __len__(self):
        return len(self.curve_dict.keys())
    
    def __getitem__(self, idx):
        curve_idx = list(self.curve_dict.keys())[idx]

        # Image processing
        curve_img_path = os.path.join(self.img_directory, f'curve_{curve_idx}.png')
        curve_img = Image.open(curve_img_path)
        curve_img = self.img_transforms(curve_img)

        #sequence processing
        sequence = self.curve_dict[curve_idx][:self.sequence_len]
        #TODO fix normalization to normalizing by mean and std of sequences in train set
        sequence = torch.tensor(sequence, dtype=torch.float32)
        sequence_normalized = (sequence - torch.tensor(self.mean, dtype=torch.float32)) / torch.tensor(self.std, dtype=torch.float32)

        #target data retrieval
        target = self.target_df.loc[self.target_df['curve_idx'] == curve_idx, 'groundtruth_target'].values[0]

        return curve_img, sequence_normalized, torch.tensor(target, dtype=torch.long), curve_idx

class FusionModel(nn.Module):
    def __init__(self, input_size, hidden_size, latent_dim, sequence_length, num_layers=5):
        super(FusionModel, self).__init__()

        self.latent_dim = latent_dim
        
        # Image processing via EfficientNet_V2_L
        self.effnet = models.efficientnet_v2_l(pretrained=True)
        num_ftrs = self.effnet.classifier[1].in_features
        self.effnet.classifier = nn.Linear(num_ftrs, self.latent_dim)  # Adjusting to output a 512-dimensional 

        # Sequence processing via LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.hidden_state = (torch.zeros(num_layers, sequence_length, hidden_size), torch.zeros(num_layers, sequence_length, hidden_size))
        
        # Final fully connected layer to ensure the LSTM output has a size of 512
        self.lstm_fc = nn.Linear(hidden_size, 512)

        # Fusion of image and sequence representations
        self.fc = nn.Sequential(
            nn.Linear(1024, 512),  # Concatenated vectors are of size 1024 (512 from image + 512 from sequence)
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, image, sequence):
        # Image processing
        img_latent = self.effnet(image)

        # Sequence processing
        lstm_out, _ = self.lstm(sequence)
        seq_latent = self.lstm_fc(lstm_out[:, -1, :])  # Taking the last output from LSTM for the whole sequence

        # Fusion
        fusion = torch.cat((img_latent, seq_latent), dim=1)
        output = self.fc(fusion)
        return output



In [12]:
device = "cuda:2" if torch.cuda.is_available() else "cpu"
print(torch.version.cuda)
print(torch.__version__)
print(device)

11.2
2.0.0.post200
cuda:2


In [13]:
with open('./data/groundtruth_df_curve_dict.pkl', 'rb') as file:
    curve_dict = pkl.load(file)
target_df = pd.read_csv('./data/groundtruth_df.csv')  # Load your DataFrame here


In [14]:
target_df.loc[:,['groundtruth_target']] = 1*(target_df.groundtruth == 1)

In [15]:
target_df[target_df.split == 'val'].curve_idx.nunique()

4632

In [16]:
# ###########################################
# ## Save curves as images
# ###########################################

# for idx in  tqdm(range(len(curve_dict.keys()))):
#     curve_idx = list(curve_dict.keys())[idx]
#     sequence = curve_dict[curve_idx][:40]

#     if not os.path.exists('../data/curve_imgs_axis/'):
#         os.makedirs('../data/curve_imgs_axis/')
#     plt.plot(sequence, linewidth=6)
#     #plt.axis('off')  # This will turn off the axis labels and ticks
#     plt.axis('on') 
#     plt.show()
#     plt.savefig(f'../data/curve_imgs_axis/curve_{curve_idx}.png')
#     plt.clf()

In [36]:
###########################################
## Get the right normalization values
###########################################

target_df_filtered = target_df[target_df['split']=='train']
curve_dict_filtered = {k: curve_dict[k] for k in curve_dict.keys() if k in target_df_filtered['curve_idx'].values}

mean_list = []
std_list = []

for key, curve in tqdm(curve_dict_filtered.items()):
    mean_curve = np.array(curve).mean().item()
    std_curve = np.array(curve).std().item()

    mean_list.append(mean_curve)
    std_list.append(std_curve)

norm_mean = np.array(mean_list).mean().item()
norm_std = np.array(std_list).mean().item()

###########################################
## Set-up data objects
###########################################

# TODO extract mean and std of sequences in train dataset

# Create Dataset and DataLoader
# train_dataset = ImageSequenceDataset(curve_dict, target_df,img_directory = '../data/curve_imgs_axis/', train=True, sequence_len=40,
#                                         mean=norm_mean, std = norm_std)
val_dataset = ImageSequenceDataset(curve_dict, target_df,img_directory = '../data/curve_imgs_axis/', split='train', sequence_len=40,
                                    mean=norm_mean, std = norm_std)
test_dataset = ImageSequenceDataset(curve_dict, target_df,img_directory = '../data/curve_imgs_axis/', split='test', sequence_len=40,
                                    mean=norm_mean, std = norm_std)

100%|██████████| 13949/13949 [00:00<00:00, 45150.65it/s]


In [37]:
# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, pin_memory=True, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, pin_memory=True, shuffle=True)

In [29]:
sequence_length = 40  # Suppose the length of your sequence is 100
input_size = 1  # Number of input features per sequence element
hidden_size = 512
latent_dim = 512
num_layers = 3
num_epoch = 10

model = FusionModel(input_size, hidden_size, latent_dim, sequence_length, num_layers=num_layers)
model.load_state_dict(torch.load('./output/fusion_model/best_model_v2.pth'))
# model = ConvLSTM(input_size=input_size, conv_out_channels=32, kernel_size=3, hidden_size=50, output_size=2, seq_len=seq_len)
model.to(device)  # If you are using GPU


/home/alpaca/anaconda3/envs/huong/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/alpaca/anaconda3/envs/huong/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_L_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_L_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FusionModel(
  (effnet): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
          )
          (stochastic_depth): StochasticDepth(p=0.0, mode=row)
        )
        (1): FusedMBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (1): BatchNorm2d(3

In [31]:
val_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(val_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out = model(images, sequences)
    val_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

100%|██████████| 145/145 [00:32<00:00,  4.49it/s]


In [34]:
val_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(val_outputs).squeeze()})
val_pred_df.to_csv('./data/fusion_pred_df.csv', index = False)

In [40]:
test_outputs = []
curve_ids = []
model.eval()
for batch in tqdm(test_loader):
    images, sequences, labels, ids = batch

    images = images.to(device)
    sequences = sequences.to(device).unsqueeze(2)
    out = model(images, sequences)
    test_outputs.append(out.detach().cpu().numpy())
    curve_ids.append(ids)

100%|██████████| 193/193 [00:41<00:00,  4.61it/s]


In [41]:
test_pred_df = pd.DataFrame({'curve_idx': np.concatenate(curve_ids), 'outputs': np.concatenate(test_outputs).squeeze()})
test_pred_df.to_csv('./data/fusion_tes_pred_df.csv', index = False)